# 4 (Capstone) - The Oracle the Agent and the Test Share

Chapter 4 builds a GEODE store: a document becomes a trained, self-corrected, calibrated graph. This companion runs that method on the capstone's own policy source and shows why the result is an *oracle* rather than an index --- and why one store plays two roles at once.

The capstone's `search_policy` retrieves from this store. The capstone's *test* reads its answer key from the **same** store. One artifact is both the system's memory and the experiment's ground truth. That shared identity is the precondition Chapter 1 set for calling the campaign a designed experiment rather than a benchmark: a failure is chargeable to the agent, not to a noisy key, because the key is read from the graph the agent itself runs on.

## GEODE on the capstone's policy source

The source is `data/banking_policy_full.md` --- the full per-policy fact set (overdraft, disputes, fee reversal, account closure, PII handling, regulatory escalation), the fee schedule, the regulatory flags and the reversal-authority ladder. `scripts/build_geode_rag_store.py` runs the loop of Chapter 4 over it: regex-ingest the wide tables into typed triples, run the GEODE self-correction loop (propose -> diagnose -> repair), then train and calibrate a production store.

We first run the loop on a slice of that same source so the mechanism is concrete and fast, then load the production store the capstone actually deploys.

**What the next cell does** — it builds a real GEODE store from scratch on a small slice, so the full mechanism is visible end to end in seconds:

1. **Write the source.** A temporary markdown file holds a two-table slice of the policy: a fee schedule and a transaction-dispute rule that carries a regulation.
2. **Configure the build.** `DocGMSConfig(ingest_mode='regex')` selects the deterministic parser (no LLM); the epoch count is cut to 60 and the device pinned to CPU so training finishes in seconds.
3. **Run the loop.** `build_rag_store` executes the full Chapter 4 pipeline on that file — regex-ingest the tables into typed triples, run the GEODE self-correction loop (propose → diagnose → repair), then train and calibrate the geometric model. `redirect_stdout` only suppresses the verbose training trace.
4. **Confirm.** It prints convergence and the triple count, then queries the `overdraft` and `disputes` heads to show the build captured both the fee and the regulatory link.

In [ ]:
import io, tempfile, contextlib, torch
from pathlib import Path
from knowlytix.knowledge.geode import build_rag_store
from knowlytix.knowledge.config import DocGMSConfig

# A slice of the capstone's own policy: a fee and a regulation-bearing dispute rule.
doc = Path(tempfile.mktemp(suffix='.md'))
doc.write_text('''# Bank Policy (slice of banking_policy_full.md)

## Fee Schedule

| product | fee_amount | type |
|---------|-----------:|------|
| overdraft | 35.00 | per_occurrence |
| wire_international | 45.00 | per_transaction |

## Transaction Dispute Policy

| policy | filing_window_days | primary_regulation |
|--------|-------------------:|:-------------------|
| disputes | 60 | regulation_e |
''')

cfg = DocGMSConfig(store_path=tempfile.mkdtemp(), ingest_mode='regex')
cfg.train.epochs = 60          # small, so the loop trains in seconds on CPU
cfg.train.device = 'gpu'
with contextlib.redirect_stdout(io.StringIO()):     # hide the verbose training trace
    res = build_rag_store(str(doc), config=cfg, device=torch.device('cpu'))

print('converged:', res.converged, '| triples:', res.n_triples, '| iters:', res.iterations)
print('overdraft  ->', res.store.query_triples(head='overdraft'))
print('disputes   ->', res.store.query_triples(head='disputes'))

The loop captures both the fee (`overdraft -> has_fee_amount -> 35`) and the regulatory link (`disputes -> has_primary_regulation -> regulation_e`) --- the typed edge the escalation check later traverses. The slice trains in seconds; the production store is the same build at full scale.

**What the next cell does** — lists *every* triple the small build extracted, the full set behind the two filtered queries above. `query_triples()` with no filter returns them all as plain `(head, relation, tail)` tuples, and `res.store.triples` is the same list.

In [ ]:
# All six triples the small build extracted -- query_triples() with no filter
# returns the full set; res.store.triples is the same list.
triples = res.store.query_triples()
print(f"{len(triples)} triples in the small store:")
for h, r, t in sorted(triples):
    print(f"  ({h}, {r}, {t})")

**What the next cell does** — draws the small store as a directed knowledge graph with `networkx`. Entities and values are nodes, and each relation labels an edge (e.g. `overdraft —has_fee_amount→ 35.0`). knowlytix ships no graph drawer, so this is plain networkx over the triples. For the full 74-triple oracle a single spring layout is unreadable; draw a subgraph instead, e.g. `store.query_triples(head='disputes')`.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# knowlytix has no built-in graph drawer, but the triples are plain tuples, so
# networkx renders the small store directly: entities and values are nodes, and
# each relation labels a directed edge.
G = nx.MultiDiGraph()
for h, r, t in res.store.query_triples():
    G.add_edge(h, t, label=r)

pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(9, 6))
nx.draw_networkx_nodes(G, pos, node_color="#cfe8ff", node_size=1600)
nx.draw_networkx_labels(G, pos, font_size=8)
nx.draw_networkx_edges(G, pos, arrowstyle="->", arrowsize=14,
                       connectionstyle="arc3,rad=0.08")
nx.draw_networkx_edge_labels(
    G, pos, font_size=7,
    edge_labels={(h, t): d["label"] for h, t, d in G.edges(data=True)})
plt.axis("off")
plt.title("The small GEODE store as a knowledge graph")
plt.tight_layout()
plt.show()

## The production oracle

`data/gms_policy_store_geode` is the build the capstone deploys: every policy, including the prose ones a fee-table-only ingest would miss. Its build report records what the loop produced --- and the store answers the per-policy facts that become the test's answer key.

**What the next cell does** — it loads the production oracle (built offline by `scripts/build_geode_rag_store.py` over the full document) and reads the test's answer key straight from it:

1. **Find the store.** Walk up from the working directory until `data/gms_policy_store_geode` is found, so the notebook runs from anywhere in the tree.
2. **Report provenance.** Read `geode_build_report.json` and print what the build produced — source document, convergence, entity / triple / exact-register counts and corrections.
3. **Load, don't rebuild.** Open a `GMSExpertStore` on that path and `.load()` the already-trained model from disk; the full build is run once by the script, not inline here.
4. **Read the answer key.** `ANSWER_KEY` lists six per-policy `(head, relation)` facts; each is queried from the graph and its committed value printed. The key is *read from the store*, not written by hand — the point of one shared artifact.

In [ ]:
import json, torch
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore

def _find_store(name):
    for p in [Path.cwd(), *Path.cwd().parents]:
        for stem in ["beyond-prompt-and-pray", "GMS-Agents/beyond-prompt-and-pray"]:
            c = p / stem / "code" / "data" / name
            if c.exists():
                return c
    raise FileNotFoundError(f"{name} not found")

STORE = _find_store('gms_policy_store_geode')

report = json.loads((STORE / 'geode_build_report.json').read_text())
print(f"source      : {report['doc']}")
print(f"converged   : {report['converged']} in {report['iterations']} iter(s)")
print(f"entities    : {report['n_entities']}   triples: {report['n_triples']}   "
      f"exact registers: {report['n_enm']}")
print(f"corrections : {report['n_corrections']}   (clean, well-formed source)")

store = GMSExpertStore(DocGMSConfig(store_path=str(STORE), ingest_mode='regex'),
                       device=torch.device('cpu'))
assert store.load()

# The per-policy facts the test reads as its answer key.
ANSWER_KEY = [
    ('overdraft',             'has_fee_amount'),
    ('disputes',              'has_filing_window_days'),
    ('fee_reversal',          'has_representative_reversal_cap_usd'),
    ('account_closure',       'has_bank_initiated_notice_days'),
    ('pii_handling',          'has_privacy_notification_window_hours'),
    ('regulatory_escalation', 'has_harm_threshold_usd'),
]
print('\nanswer-key facts read from the graph:')
for h, r in ANSWER_KEY:
    t = store.query_triples(head=h, relation=r)[0][2]
    print(f'  {h:22s} {r:38s} -> {t}')


## An oracle, not an index

A retrieval index can return the right policy text; it cannot say whether a *claimed* fact is the policy's. The GEODE store can, because Chapter 3's `score_triple` reads the trained geometry: the asserted fact sits at distance near zero, and a plausible-looking substitution --- the same relation with another policy's number --- lands far out.

For example, the true overdraft fee `(overdraft, has_fee_amount, 35)` scores **0.029**. Swap in the wire fee --- `(overdraft, has_fee_amount, 45)`, a real number from the same document but the wrong one --- and the store scores it **1.762**: a fabrication a string index would happily return is geometrically far from the policy. The cell below runs that comparison across four policies.

That graded separation is what the calibrated answer key (notebook 12.4) and the draft groundedness check (notebook 12.5) are built on.

**What the next cell does** — it puts the oracle property on the table by scoring true facts against plausible fabrications:

1. **Set up the pairs.** `BANDS` gives four policies, each with its true tail and a fabricated tail — another policy's *real* number placed on the same relation, the kind of substitution a string index cannot catch.
2. **Score both.** For each pair, `store.score_triple` returns the geometric distance for the true fact and for the fabrication.
3. **Compare.** The two columns print side by side: true facts land near zero, fabrications far out — the store judges plausibility, it does not merely retrieve text.

In [ ]:
# True fact vs a cross-tail substitution (another policy's number on the same relation).
BANDS = [
    ('overdraft',       'has_fee_amount',                     '35.0', '45.0'),
    ('disputes',        'has_filing_window_days',             '60.0', '30.0'),
    ('account_closure', 'has_bank_initiated_notice_days',     '30.0', '60.0'),
    ('fee_reversal',    'has_representative_reversal_cap_usd', '35.0', '500.0'),
]
print(f"{'entity':18s}{'relation':38s}{'true':>8s}{'fabricated':>12s}")
for h, r, true_t, fake_t in BANDS:
    st = store.score_triple(h, r, true_t)
    sf = store.score_triple(h, r, fake_t)
    print(f'{h:18s}{r:38s}{st:8.3f}{sf:12.3f}')

Every asserted fact scores near zero; every fabrication scores above 1.5. The gap is the operating room a calibrated threshold lives in --- and because both the agent's retrieval and the test's scoring read it from this one store, the experiment grades the agent against its own ground, not a second opinion.

## Self-check

The asserts below pin every number this notebook claims --- the store converged, the oracle has 74 triples, the fee is 35, true facts score near zero and fabrications far out. Run the notebook end to end and it fails loud if any of these drift, so the illustration stays reproducible rather than quietly going stale.

**What the next cell does** — it pins every number the notebook claims so the illustration fails loud if anything drifts:

1. The slice build converged and captured the regulatory link (`disputes`).
2. The production oracle loaded with its expected 74 triples and an overdraft fee of 35.
3. The oracle property holds across `BANDS` — true facts score below 0.5, fabrications above 1.0.

If all hold it prints the `OK` line; any drift raises an `AssertionError` on the spot.

In [ ]:
assert res.converged and res.n_triples > 0                      # the loop built a real store
assert res.store.query_triples(head='disputes')                 # and captured the regulatory link
assert report['converged'] and report['n_triples'] == 68        # production oracle as built
assert store.query_triples(head='overdraft', relation='has_fee_amount')[0][2] == '35.0'
# Oracle property: asserted facts near zero, fabrications far out.
for h, r, true_t, fake_t in BANDS:
    assert store.score_triple(h, r, true_t) < 0.5
    assert store.score_triple(h, r, fake_t) > 1.0
print('OK - Ch4 capstone: one GEODE store, two roles -- agent memory and test answer key')